In [ ]:
import os
os.environ["SB_USERNAME"] = "<TU_USUARIO_STATSBOMB>"
os.environ["SB_PASSWORD"] = "<TU_PASSWORD_STATSBOMB>"

from statsbombpy import sb
events_df = sb.events(match_id=3799451)  # ya no hace falta pasar creds

In [ ]:
from statsbombpy import sb

comps = sb.competitions()
liga_mx = comps[comps.competition_name.str.contains("Liga MX", case=False, na=False)]
print(liga_mx[["competition_id", "season_id", "season_name"]])

In [ ]:
import pandas as pd

comp_id = liga_mx.competition_id.iloc[0]

all_matches = []
for season_id in liga_mx.season_id.tolist():
    matches = sb.matches(competition_id=comp_id, season_id=season_id)
    matches["season_id"] = season_id
    all_matches.append(matches)

matches_df = pd.concat(all_matches, ignore_index=True)

# Ver qué entrenadores aparecen
print(matches_df[["match_id", "home_team", "home_managers", "away_team", "away_managers", "season_id"]].head(20))

In [ ]:
# Ver cuántos partidos hay por temporada
print(matches_df.groupby("season_id").size())

# Confirmar que están las temporadas que esperabas
print(matches_df.season_id.unique())
print(f"Total de partidos en todas las temporadas: {len(matches_df)}")

In [ ]:
# Ver si hay match_id repetidos (señal de duplicados)
print(f"Total de filas: {len(matches_df)}")
print(f"Match_id únicos: {matches_df.match_id.nunique()}")

# Ver las fechas/torneos dentro de esas dos temporadas grandes
print(matches_df[matches_df.season_id.isin([317, 318])][["match_date", "competition_stage"]].drop_duplicates().sort_values("match_date").head(20))

In [ ]:
# Ver cuáles match_id están duplicados
duplicados = matches_df[matches_df.duplicated(subset="match_id", keep=False)].sort_values("match_id")
print(duplicados[["match_id", "season_id", "competition_stage", "home_team", "away_team", "match_date"]].head(20))

# Ver si los duplicados tienen el mismo season_id o distinto
print(duplicados.groupby("match_id")["season_id"].nunique().value_counts())

In [ ]:
# Ver la tabla de competiciones filtrada, tal cual antes de iterar
print(liga_mx[["competition_id", "season_id", "season_name", "competition_gender"]])

# Ver si hay season_id repetidos ahí mismo
print(liga_mx.season_id.value_counts())

In [ ]:
# Quedarte solo con Liga MX varonil (o el género que te interese) para evitar duplicados
liga_mx = liga_mx[liga_mx.competition_gender == "male"]

In [ ]:
# Eliminar duplicados de season_id antes de iterar
liga_mx = liga_mx.drop_duplicates(subset="season_id")

In [ ]:
matches_df = matches_df.drop_duplicates(subset="match_id").reset_index(drop=True)
print(f"Total de partidos después de limpiar: {len(matches_df)}")

In [ ]:
# Ver cobertura del dato de entrenador
print(f"Partidos con home_manager: {matches_df.home_managers.notna().sum()} de {len(matches_df)}")
print(f"Partidos con away_manager: {matches_df.away_managers.notna().sum()} de {len(matches_df)}")

# Lista de entrenadores únicos disponibles
managers_home = matches_df.home_managers.dropna().unique()
managers_away = matches_df.away_managers.dropna().unique()
todos_managers = sorted(set(managers_home) | set(managers_away))
print(todos_managers)

In [ ]:
vacios_home = (matches_df.home_managers == "").sum()
vacios_away = (matches_df.away_managers == "").sum()
print(f"Partidos con home_manager vacío: {vacios_home}")
print(f"Partidos con away_manager vacío: {vacios_away}")

In [ ]:
# Función para contar partidos reales por entrenador, separando los campos con coma
def contar_partidos_por_entrenador(matches_df):
    registros = []
    for _, row in matches_df.iterrows():
        home_mgrs = [m.strip() for m in str(row.home_managers).split(",") if m.strip()]
        away_mgrs = [m.strip() for m in str(row.away_managers).split(",") if m.strip()]
        for mgr in home_mgrs:
            registros.append({"entrenador": mgr, "match_id": row.match_id, "equipo": row.home_team})
        for mgr in away_mgrs:
            registros.append({"entrenador": mgr, "match_id": row.match_id, "equipo": row.away_team})

    df_entrenadores = pd.DataFrame(registros)
    conteo = df_entrenadores.groupby("entrenador").agg(
        n_partidos=("match_id", "nunique"),
        equipos=("equipo", lambda s: sorted(s.unique()))
    ).sort_values("n_partidos", ascending=False)
    return conteo

conteo_entrenadores = contar_partidos_por_entrenador(matches_df)
print(conteo_entrenadores.head(30))

In [ ]:
def contar_partidos_por_entrenador_y_equipo(matches_df):
    registros = []
    for _, row in matches_df.iterrows():
        home_mgrs = [m.strip() for m in str(row.home_managers).split(",") if m.strip()]
        away_mgrs = [m.strip() for m in str(row.away_managers).split(",") if m.strip()]
        for mgr in home_mgrs:
            registros.append({"entrenador": mgr, "match_id": row.match_id, "equipo": row.home_team, "season_id": row.season_id})
        for mgr in away_mgrs:
            registros.append({"entrenador": mgr, "match_id": row.match_id, "equipo": row.away_team, "season_id": row.season_id})

    df_entrenadores = pd.DataFrame(registros)

    conteo = (
        df_entrenadores.groupby(["entrenador", "equipo"])
        .agg(
            n_partidos=("match_id", "nunique"),
            temporadas=("season_id", lambda s: sorted(s.unique())),
        )
        .reset_index()
        .sort_values(["entrenador", "n_partidos"], ascending=[True, False])
    )
    return conteo

conteo_entrenador_equipo = contar_partidos_por_entrenador_y_equipo(matches_df)
print(conteo_entrenador_equipo)

# Para ver solo un entrenador específico:
print(conteo_entrenador_equipo[conteo_entrenador_equipo.entrenador == "Nombre del entrenador"])

# Para ordenar por volumen total, viendo primero a los que más partidos tienen:
print(conteo_entrenador_equipo.sort_values("n_partidos", ascending=False).head(30))

In [ ]:
mapa_temporadas = liga_mx.set_index("season_id")["season_name"].to_dict()

def contar_partidos_por_entrenador_y_equipo(matches_df, mapa_temporadas):
    registros = []
    for _, row in matches_df.iterrows():
        home_mgrs = [m.strip() for m in str(row.home_managers).split(",") if m.strip()]
        away_mgrs = [m.strip() for m in str(row.away_managers).split(",") if m.strip()]
        temporada = mapa_temporadas.get(row.season_id, row.season_id)
        for mgr in home_mgrs:
            registros.append({"entrenador": mgr, "match_id": row.match_id, "equipo": row.home_team, "temporada": temporada})
        for mgr in away_mgrs:
            registros.append({"entrenador": mgr, "match_id": row.match_id, "equipo": row.away_team, "temporada": temporada})

    df_entrenadores = pd.DataFrame(registros)

    conteo = (
        df_entrenadores.groupby(["entrenador", "equipo"])
        .agg(
            n_partidos=("match_id", "nunique"),
            temporadas=("temporada", lambda s: sorted(s.unique())),
        )
        .reset_index()
        .sort_values(["entrenador", "n_partidos"], ascending=[True, False])
    )
    return conteo

conteo_entrenador_equipo = contar_partidos_por_entrenador_y_equipo(matches_df, mapa_temporadas)
print(conteo_entrenador_equipo)

In [ ]:
# Ver el top completo ordenado por número de partidos (el candidato más fuerte primero)
print(conteo_entrenador_equipo.sort_values("n_partidos", ascending=False).head(20))

In [ ]:
def obtener_matches_entrenador_equipo(matches_df, entrenador, equipo):
    registros = []
    for _, row in matches_df.iterrows():
        home_mgrs = [m.strip() for m in str(row.home_managers).split(",") if m.strip()]
        away_mgrs = [m.strip() for m in str(row.away_managers).split(",") if m.strip()]

        if entrenador in home_mgrs and row.home_team == equipo:
            registros.append(row)
        elif entrenador in away_mgrs and row.away_team == equipo:
            registros.append(row)

    return pd.DataFrame(registros).reset_index(drop=True)

partidos_jardine_america = obtener_matches_entrenador_equipo(
    matches_df, "André Soares Jardine", "América"
)

print(f"Total de partidos: {len(partidos_jardine_america)}")
print(partidos_jardine_america[["match_id", "season_id", "match_date", "home_team", "away_team"]].sort_values("match_date"))

In [ ]:
print(partidos_jardine_america.groupby("season_id").agg(
    n_partidos=("match_id", "count"),
    primera_fecha=("match_date", "min"),
    ultima_fecha=("match_date", "max"),
))

In [ ]:
from statsbombpy import sb
import pandas as pd

all_events = []
match_ids = partidos_jardine_america.match_id.tolist()

for i, match_id in enumerate(match_ids):
    events = sb.events(match_id=match_id)
    events["match_id"] = match_id
    all_events.append(events)
    if (i + 1) % 10 == 0:
        print(f"Procesados {i + 1}/{len(match_ids)} partidos")

events_jardine = pd.concat(all_events, ignore_index=True)
print(f"Total de eventos: {len(events_jardine)}")
print(f"Columnas disponibles: {events_jardine.shape[1]}")

In [ ]:
pip install pyarrow

In [ ]:
events_jardine.to_csv("events_jardine_america.csv", index=False)
print("Guardado correctamente.")

In [ ]:
from framework_analitico import (
    build_possessions,
    extract_offensive_sequences,
    identify_transitions,
    identify_pressure_phases,
)

posesiones = build_possessions(events_jardine)
print(f"Total de posesiones: {len(posesiones)}")

secuencias = extract_offensive_sequences(events_jardine)
print(f"Secuencias ofensivas (≥3 pases): {len(secuencias)}")

fases_presion = identify_pressure_phases(events_jardine)
print(f"Total de fases de presión: {len(fases_presion)}")

In [ ]:
# Solo posesiones de América
posesiones_america = posesiones[posesiones.team == "América"]
print(f"Posesiones de América: {len(posesiones_america)}")

# Solo fases de presión de América
presion_america = fases_presion[fases_presion.team == "América"]
print(f"Fases de presión de América: {len(presion_america)}")

# Comparar con el rival, para tener un punto de referencia
presion_rival = fases_presion[fases_presion.team != "América"]
print(f"Fases de presión promedio de rivales: {presion_rival.groupby('match_id').size().mean():.1f} por partido")
print(f"Fases de presión promedio de América: {presion_america.groupby('match_id').size().mean():.1f} por partido")

In [ ]:
# Normalizar: fases de presión por cada posesión rival (osea, por cada oportunidad de presionar)
posesiones_rival = posesiones[posesiones.team != "América"]

presion_por_oportunidad_america = presion_america.groupby("match_id").size() / posesiones_rival.groupby("match_id").size()
presion_por_oportunidad_rival = presion_rival.groupby("match_id").size() / posesiones_america.groupby("match_id").size()

print(f"Presión de América por posesión rival: {presion_por_oportunidad_america.mean():.2f}")
print(f"Presión de rivales por posesión de América: {presion_por_oportunidad_rival.mean():.2f}")

In [ ]:
pip install mplsoccer

In [ ]:
import pandas as pd

# Filtrar solo eventos de presión de América
pressure_events = events_jardine[
    (events_jardine.type == "Pressure") & (events_jardine.team == "América")
].copy()

# La columna location viene como [x, y] - la separamos
pressure_events[["x", "y"]] = pd.DataFrame(
    pressure_events["location"].tolist(), index=pressure_events.index
)

print(pressure_events[["x", "y"]].describe())

In [ ]:
from mplsoccer import Pitch
import matplotlib.pyplot as plt

pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="black")
fig, ax = pitch.draw(figsize=(10, 7))

pitch.kdeplot(
    pressure_events.x, pressure_events.y,
    ax=ax,
    cmap="Reds",
    fill=True,
    levels=100,
    alpha=0.7,
)

ax.set_title("Mapa de calor de presión — América (Jardine)", fontsize=14)
plt.savefig("mapa_presion_america.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
pressure_rivales = events_jardine[
    (events_jardine.type == "Pressure") & (events_jardine.team != "América")
].copy()
pressure_rivales[["x", "y"]] = pd.DataFrame(
    pressure_rivales["location"].tolist(), index=pressure_rivales.index
)

fig, axes = pitch.grid(figheight=8, ncols=2, title_height=0.08, endnote_height=0)

pitch.kdeplot(pressure_events.x, pressure_events.y, ax=axes["pitch"][0], cmap="Reds", fill=True, levels=100)
axes["pitch"][0].set_title("América presiona")

pitch.kdeplot(pressure_rivales.x, pressure_rivales.y, ax=axes["pitch"][1], cmap="Blues", fill=True, levels=100)
axes["pitch"][1].set_title("Rivales presionan")

plt.savefig("comparacion_presion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

def normalizar_direccion_ataque(events_df, equipo="América"):
    """
    Voltea las coordenadas x, y para que el equipo especificado
    siempre 'ataque' hacia x=120, sin importar el periodo o si
    jugó de local/visitante.

    Usa como referencia los tiros del equipo: si el promedio de x
    de sus tiros en un periodo dado es menor a 60 (mitad de cancha),
    asumimos que ese periodo estaba invertido y volteamos.
    """
    df = events_df.copy()
    df[["x", "y"]] = pd.DataFrame(df["location"].tolist(), index=df.index)

    # Referencia: tiros del equipo por partido y periodo
    tiros = df[(df.team == equipo) & (df.type == "Shot")]
    referencia = tiros.groupby(["match_id", "period"])["x"].mean().reset_index()
    referencia["atacando_derecha"] = referencia["x"] > 60

    df = df.merge(referencia[["match_id", "period", "atacando_derecha"]], on=["match_id", "period"], how="left")

    # Si no hay tiros en ese periodo para inferir dirección, asumimos que sí ataca a la derecha (no se voltea)
    df["atacando_derecha"] = df["atacando_derecha"].fillna(True)

    volteo = ~df["atacando_derecha"]
    df.loc[volteo, "x"] = 120 - df.loc[volteo, "x"]
    df.loc[volteo, "y"] = 80 - df.loc[volteo, "y"]

    return df

events_normalizados = normalizar_direccion_ataque(events_jardine, equipo="América")

pressure_events = events_normalizados[
    (events_normalizados.type == "Pressure") & (events_normalizados.team == "América")
]

print(pressure_events[["x", "y"]].describe())

In [ ]:
# 1. Normalizar dirección de ataque (necesario antes de field tilt o mapas de calor)
events_normalizados = normalizar_direccion_ataque(events_jardine, equipo="América")

# 2. Perfil ofensivo/defensivo completo
perfil = perfil_ofensivo_defensivo(events_normalizados, equipo="América")
print(perfil)

# 3. Clasificación de intensidad de presión, específica para América
fases_clasificadas, cortes = clasificar_intensidad_presion(fases_presion, equipo="América")
print(f"Cortes (baja/alta): {cortes}")
print(fases_clasificadas.intensidad.value_counts())